# GitHub PR Tracker - Backend Technical Requirements

**Project:** GitHub PR Tracker Dashboard  
**Document:** Backend Technical Requirements  
**Version:** 1.0 (Phase 1)  
**Date:** 2 June 2026  
**Tech Stack:** Python + FastAPI

## 1. Tech Stack Summary

| Component | Technology | Version | Purpose |
|-----------|-----------|---------|--------|
| **Language** | Python | 3.11+ | Core runtime |
| **Framework** | FastAPI | 0.100+ | REST API framework (async, auto-docs) |
| **HTTP Client** | httpx | 0.27+ | Async HTTP client for GitHub API calls |
| **GitHub SDK** | PyGithub or httpx (direct) | — | GitHub API integration |
| **Server** | Uvicorn | 0.30+ | ASGI server |
| **Validation** | Pydantic v2 | 2.0+ | Request/response models |
| **Environment** | python-dotenv | — | Configuration management |
| **Testing** | pytest + pytest-asyncio | — | Unit and integration tests |
| **Linting** | Ruff | — | Fast linter + formatter |

> **Note (Phase 1):** No in-memory caching layer. All requests fetch fresh data from GitHub API.  
> With ~10 team members and 5,000 req/hr rate limit, caching is unnecessary for Phase 1.  
> Can be revisited in Phase 2 if rate limits become a concern.

## 2. Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                     Frontend (React + Vite)                   │
└─────────────────────────┬───────────────────────────────────┘
                          │ REST API (JSON)
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   FastAPI Backend                             │
│                                                              │
│  ┌──────────┐  ┌──────────────┐  ┌───────────────────────┐ │
│  │  Routes  │  │  Services    │  │  GitHub Client        │ │
│  │  (API)   │──│  (Business   │──│  (httpx + GitHub API) │ │
│  │          │  │   Logic)     │  │                       │ │
│  └──────────┘  └──────────────┘  └───────────────────────┘ │
│                                                              │
│  ┌──────────────────────────────────────────────────────────┐│
│  │  CODEOWNERS Parser (Glob matching)                        ││
│  └──────────────────────────────────────────────────────────┘│
└─────────────────────────┬───────────────────────────────────┘
                          │ HTTPS
                          ▼
┌─────────────────────────────────────────────────────────────┐
│                   GitHub REST API (api.github.com)            │
└─────────────────────────────────────────────────────────────┘
```

**Key:** No caching layer in Phase 1. Backend always fetches fresh data from GitHub on each request.

## 3. Project Structure

```
backend/
├── app/
│   ├── __init__.py
│   ├── main.py                 # FastAPI app entry point
│   ├── config.py               # Configuration (env vars, settings)
│   ├── routes/
│   │   ├── __init__.py
│   │   ├── pull_requests.py    # PR endpoints
│   │   ├── team.py             # Team member endpoints
│   │   └── health.py           # Health check
│   ├── services/
│   │   ├── __init__.py
│   │   ├── github_client.py    # GitHub API wrapper (httpx)
│   │   ├── pr_service.py       # PR business logic (fetch, filter, enrich)
│   │   ├── team_service.py     # Team member resolution
│   │   └── codeowners.py       # CODEOWNERS parser + matcher
│   ├── models/
│   │   ├── __init__.py
│   │   ├── pr.py               # PR Pydantic models
│   │   ├── team.py             # Team member models
│   │   └── config.py           # Config models
│   └── utils/
│       ├── __init__.py
│       ├── time_helpers.py     # Age calculation utilities
│       └── glob_matcher.py     # CODEOWNERS glob pattern matching
├── tests/
│   ├── __init__.py
│   ├── test_pr_service.py
│   ├── test_codeowners.py
│   ├── test_team_service.py
│   └── test_routes.py
├── .env.example                # Environment variable template
├── requirements.txt            # Python dependencies
├── pyproject.toml              # Project metadata + tool config
└── Dockerfile                  # Container build (Linux target)
```

## 4. API Endpoints

### 4.1 Pull Requests

| Method | Endpoint | Description | Query Params |
|--------|----------|-------------|-------------|
| `GET` | `/api/v1/pull-requests` | Fetch all open PRs for the team | `branch_type` (all/main/feature), `sort_by` (age/author/reviewers), `sort_order` (asc/desc) |
| `GET` | `/api/v1/pull-requests/{pr_number}` | Fetch detailed info for a single PR | — |

### 4.2 Team

| Method | Endpoint | Description | Query Params |
|--------|----------|-------------|-------------|
| `GET` | `/api/v1/team/members` | Get list of resolved team members | — |

### 4.3 Health

| Method | Endpoint | Description |
|--------|----------|-------------|
| `GET` | `/api/v1/health` | Health check (includes GitHub API rate limit status) |

## 5. API Response Models

### 5.1 GET `/api/v1/pull-requests` Response

```json
{
  "total_count": 12,
  "filters_applied": {
    "branch_type": "all",
    "sort_by": "age",
    "sort_order": "desc"
  },
  "pull_requests": [
    {
      "number": 142,
      "title": "Add user authentication module",
      "author": {
        "username": "johndoe",
        "avatar_url": "https://avatars.githubusercontent.com/u/12345"
      },
      "base_branch": "main",
      "head_branch": "feature/auth-module",
      "branch_type": "main",
      "created_at": "2026-05-29T10:30:00Z",
      "age": {
        "days": 4,
        "hours": 2,
        "display": "4d 2h",
        "is_stale": true,
        "threshold_days": 3
      },
      "active_reviewers_count": 2,
      "code_owner_status": {
        "required": true,
        "approved": false
      },
      "html_url": "https://github.com/org/repo/pull/142",
      "labels": ["enhancement", "backend"]
    }
  ]
}
```

### 5.2 GET `/api/v1/pull-requests/{pr_number}` Response (Detail)

```json
{
  "number": 142,
  "title": "Add user authentication module",
  "body": "## Summary\nThis PR adds the authentication module...",
  "author": {
    "username": "johndoe",
    "avatar_url": "https://avatars.githubusercontent.com/u/12345"
  },
  "base_branch": "main",
  "head_branch": "feature/auth-module",
  "branch_type": "main",
  "created_at": "2026-05-29T10:30:00Z",
  "age": {
    "days": 4,
    "hours": 2,
    "display": "4d 2h",
    "is_stale": true,
    "threshold_days": 3
  },
  "active_reviewers": [
    {
      "username": "reviewer1",
      "avatar_url": "https://avatars.githubusercontent.com/u/111",
      "state": "APPROVED"
    },
    {
      "username": "reviewer2",
      "avatar_url": "https://avatars.githubusercontent.com/u/222",
      "state": "COMMENTED"
    }
  ],
  "active_reviewers_count": 2,
  "code_owner_status": {
    "required": true,
    "approved": false,
    "owners": [
      { "username": "codeowner1", "has_approved": false },
      { "username": "codeowner2", "has_approved": false }
    ]
  },
  "files_changed": {
    "total": 8,
    "additions": 245,
    "deletions": 32
  },
  "labels": ["enhancement", "backend"],
  "html_url": "https://github.com/org/repo/pull/142"
}
```

## 6. GitHub API Integration

### 6.1 APIs Used

| # | GitHub API Endpoint | Purpose |
|---|--------------------|---------|
| 1 | `GET /repos/{owner}/{repo}/pulls?state=open` | Fetch open PRs (handles pagination server-side) |
| 2 | `GET /orgs/{org}/teams/{team_slug}/members` | Fetch team members |
| 3 | `GET /repos/{owner}/{repo}/pulls/{number}/reviews` | Fetch PR reviews |
| 4 | `GET /repos/{owner}/{repo}/pulls/{number}/comments` | Fetch PR comments |
| 5 | `GET /repos/{owner}/{repo}/pulls/{number}/files` | Fetch changed files |
| 6 | `GET /repos/{owner}/{repo}/contents/.github/CODEOWNERS` | Fetch CODEOWNERS file |
| 7 | `GET /rate_limit` | Check API rate limit |

### 6.2 Rate Limiting (Phase 1 Decision)

- GitHub API allows **5,000 requests/hour** for authenticated requests.
- With ~10 team members and a small number of PRs, we will **never approach** this limit.
- **No caching / TTL in Phase 1** — all data fetched fresh on each API call.
- Rate limit info is still exposed via the `/api/v1/health` endpoint for observability.
- **Phase 2 consideration:** If usage grows significantly, add in-memory TTL caching (e.g., `cachetools`) for team members and CODEOWNERS.

### 6.3 Pagination (Server-side)

- GitHub returns max 30 PRs per page by default (configurable up to 100).
- Backend handles pagination internally using `per_page=100` and `Link` header / page iteration.
- Frontend receives a single consolidated response (no client-side pagination needed in Phase 1).

### 6.4 Authentication

- GitHub PAT (Personal Access Token) passed via `Authorization: Bearer {token}` header.
- Token stored in environment variable `GITHUB_TOKEN`.
- Required scopes: `repo`, `read:org`.

## 7. Core Services Logic

### 7.1 PR Service (`pr_service.py`)

```python
# Pseudocode flow for fetching dashboard data

async def get_team_pull_requests(branch_type, sort_by, sort_order):
    # 1. Get team members
    team_members = await team_service.get_members()
    
    # 2. Fetch all open PRs
    all_prs = await github_client.get_open_prs()
    
    # 3. Filter by team members
    team_prs = [pr for pr in all_prs if pr.author in team_members]
    
    # 4. Filter by branch type
    if branch_type != "all":
        team_prs = filter_by_branch_type(team_prs, branch_type)
    
    # 5. Enrich each PR with:
    for pr in team_prs:
        pr.age = calculate_age(pr.created_at)
        pr.is_stale = check_staleness(pr.age, pr.branch_type)
        pr.active_reviewers_count = await get_active_reviewer_count(pr)
        pr.code_owner_status = await check_code_owner_approval(pr)
    
    # 6. Sort
    team_prs = sort_prs(team_prs, sort_by, sort_order)
    
    return team_prs
```

### 7.2 Active Reviewer Logic

```python
async def get_active_reviewer_count(pr):
    reviews = await github_client.get_pr_reviews(pr.number)
    comments = await github_client.get_pr_comments(pr.number)
    
    # Unique users who have reviewed (APPROVED, CHANGES_REQUESTED)
    # or commented (excluding the PR author)
    active_users = set()
    
    for review in reviews:
        if review.state in ("APPROVED", "CHANGES_REQUESTED"):
            active_users.add(review.user)
    
    for comment in comments:
        if comment.user != pr.author:
            active_users.add(comment.user)
    
    return len(active_users)
```

### 7.3 Staleness Check

```python
def check_staleness(age_days: float, branch_type: str) -> bool:
    thresholds = {
        "main": config.AGING_THRESHOLD_MAIN,      # default: 3 days
        "feature": config.AGING_THRESHOLD_FEATURE  # default: 2 days
    }
    return age_days >= thresholds.get(branch_type, 3)
```

## 8. CODEOWNERS Parser

### 8.1 Parsing Logic

```python
# CODEOWNERS file format:
# <pattern> <owner1> <owner2> ...
# Examples:
#   *           @org/global-owners
#   *.js        @frontend-team
#   /src/api/   @backend-team @user1

def parse_codeowners(content: str) -> list[CodeOwnerRule]:
    rules = []
    for line in content.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        pattern = parts[0]
        owners = parts[1:]  # @user or @org/team
        rules.append(CodeOwnerRule(pattern=pattern, owners=owners))
    return rules
```

### 8.2 File Matching

- Use `fnmatch` or `pathlib.PurePath.match()` for glob pattern matching.
- Last matching rule wins (same as GitHub's behavior).
- Support patterns: `*`, `**`, `/path/`, `*.ext`.

### 8.3 Fetch Strategy

- CODEOWNERS fetched fresh on each PR detail request (no caching in Phase 1).
- The file rarely changes, so the API overhead is minimal (1 extra call per detail view).
- Team member resolution for `@org/team` owners uses the same team API call.

## 9. Configuration

### Environment Variables (`.env`)

```env
# GitHub
GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxx
GITHUB_ORG=my-org
GITHUB_REPO=my-repo
GITHUB_TEAM_SLUG=phoenix

# Aging Thresholds (days)
AGING_THRESHOLD_MAIN=3
AGING_THRESHOLD_FEATURE=2

# Server
HOST=0.0.0.0
PORT=8000
CORS_ORIGINS=http://localhost:5173

# Optional: Fallback team members (comma-separated)
FALLBACK_TEAM_MEMBERS=user1,user2,user3
```

> **Removed from Phase 1:** `CACHE_TTL_*` variables (no caching layer).

## 10. Error Handling

| Scenario | Handling |
|----------|----------|
| GitHub API rate limit exceeded | Return 429 with message; log warning (unlikely in Phase 1) |
| GitHub token invalid/expired | Return 401 with clear error message |
| Team API fails (permissions) | Fall back to `FALLBACK_TEAM_MEMBERS` config |
| CODEOWNERS file not found | Skip code owner status; return `code_owner_status: null` |
| Network timeout to GitHub | Retry once with exponential backoff; then return error |
| Invalid PR number | Return 404 |

## 11. CORS Configuration

Since frontend (Vite on port 5173) and backend (FastAPI on port 8000) run on different ports during development:

```python
from fastapi.middleware.cors import CORSMiddleware

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],  # Vite dev server
    allow_credentials=True,
    allow_methods=["GET"],
    allow_headers=["*"],
)
```

## 12. Performance Considerations

| Concern | Solution |
|---------|----------|
| Multiple GitHub API calls per PR (reviews, comments, files) | Use `asyncio.gather()` for parallel requests |
| Large number of PRs | Paginate GitHub API calls internally (`per_page=100`); process in batches |
| CODEOWNERS parsing for every PR | Parse CODEOWNERS once per request cycle; reuse parsed result across PR evaluations |
| Cold start | No pre-warming needed (no cache layer) |

> **Phase 2 consideration:** If latency becomes noticeable, add TTL-based in-memory caching for team members (changes rarely) and CODEOWNERS file.

## 13. Testing Strategy

| Test Type | Tools | Coverage |
|-----------|-------|----------|
| **Unit tests** | pytest | Services, CODEOWNERS parser, age calculation, glob matcher |
| **Integration tests** | pytest + httpx (TestClient) | API endpoints with mocked GitHub responses |
| **Mocking** | pytest-mock / respx | Mock GitHub API responses |

Key test cases:
- CODEOWNERS parser handles all pattern types
- Active reviewer count excludes non-active reviewers
- Branch classification is correct
- Staleness thresholds apply correctly per branch type
- Fallback works when team API fails

## 14. Deployment

### Target: Linux Machine (Team-wide access)

The application will be deployed on a Linux server to be accessible by all team members.

### Dockerfile

```dockerfile
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY app/ ./app/

EXPOSE 8000
CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### Running Locally (Development)

```bash
# Install dependencies
pip install -r requirements.txt

# Run server
uvicorn app.main:app --reload --port 8000

# API docs available at:
# http://localhost:8000/docs (Swagger)
# http://localhost:8000/redoc (ReDoc)
```

### Production Deployment (Linux)

```bash
# Build Docker image
docker build -t pr-tracker-backend .

# Run container
docker run -d \
  --name pr-tracker-backend \
  -p 8000:8000 \
  --env-file .env \
  --restart unless-stopped \
  pr-tracker-backend

# Or run directly with systemd (alternative)
# Create /etc/systemd/system/pr-tracker-backend.service
```

### docker-compose.yml (Full Stack)

```yaml
version: "3.8"
services:
  backend:
    build: ./backend
    ports:
      - "8000:8000"
    env_file: ./backend/.env
    restart: unless-stopped
    
  frontend:
    build: ./frontend
    ports:
      - "80:80"
    depends_on:
      - backend
    restart: unless-stopped
```

## 15. Dependencies (`requirements.txt`)

```
fastapi>=0.100.0
uvicorn[standard]>=0.30.0
httpx>=0.27.0
pydantic>=2.0.0
pydantic-settings>=2.0.0
python-dotenv>=1.0.0

# Dev dependencies
pytest>=7.0.0
pytest-asyncio>=0.21.0
respx>=0.20.0
ruff>=0.1.0
```

> **Removed from Phase 1:** `cachetools` (no caching layer needed)